# Défi quotidien : Construire des insights fiables avec BERT

Ce notebook regroupe la tokenisation, l'attention, l'ajustement fin des transformateurs et l'évaluation.
Nous allons :

1. Charger et inspecter le jeu de données `tweet_eval` (sentiment).
2. Construire un pipeline de tokenisation avec `AutoTokenizer`.
3. Ajuster finement un `DistilBERT` pour la classification de sentiment (3 classes).
4. Évaluer le modèle (accuracy, macro F1) et étudier sa calibration.
5. Inspecter les cartes d'attention de la dernière couche.
6. Livrer une fonction `analyze_text()` de style production retournant l'étiquette prédite,
   la confiance, et les jetons les plus contributifs (preuve).

> **Remarque environnement :** ce notebook est conçu pour s'exécuter sur Google Colab
> (idéalement avec un runtime GPU : `Runtime > Change runtime type > GPU`). Le fine-tuning
> complet (3 époques sur ~45 000 tweets) prend plusieurs minutes sur GPU.

> **Note pour les débutants :** chaque cellule de code contient un commentaire (précédé de `#`)
> au-dessus de presque chaque ligne, pour expliquer ce qu'elle fait et pourquoi. Les commentaires
> ne sont jamais exécutés par Python — ils sont uniquement là pour vous, le lecteur.

In [ ]:
# Cette ligne installe (ou met à jour) les bibliothèques Python dont on a besoin.
# Le "!" au début dit à Jupyter/Colab : "ce n'est pas du code Python, exécute cette commande
# comme dans un terminal". pip est l'outil standard pour installer des paquets Python.
# -q = "quiet" (affiche moins de texte), -U = "upgrade" (installe la dernière version compatible).
# Chaque nom entre guillemets est une bibliothèque, avec une contrainte de version minimale (>=).
!pip install -q -U "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" "evaluate" "scikit-learn" "matplotlib" "huggingface_hub>=0.23"

In [ ]:
# On importe (= on rend disponibles dans ce notebook) les bibliothèques nécessaires.
# "import X" charge la bibliothèque X et permet d'utiliser X.quelque_chose() plus bas.

# "random" sert à générer des choix aléatoires reproductibles (utile pour le mélange des données).
import random
# "collections" fournit des outils pratiques, ici Counter pour compter des occurrences.
import collections

# "numpy" (souvent abrégé "np") est LA bibliothèque de calcul numérique en Python : tableaux,
# moyennes, etc. On l'importe sous le nom court "np" par convention (tout le monde fait pareil).
import numpy as np
# "torch" est PyTorch, la bibliothèque de deep learning qu'on utilise pour manipuler des tenseurs
# (des tableaux de nombres optimisés, similaires à des tableaux numpy mais utilisables sur GPU).
import torch
# "matplotlib.pyplot" sert à dessiner des graphiques (histogrammes, courbes, etc.).
# On l'importe sous le nom court "plt", convention universelle en Python.
import matplotlib.pyplot as plt

# "datasets" est la bibliothèque Hugging Face qui permet de télécharger et manipuler facilement
# des jeux de données publics (comme tweet_eval, qu'on va utiliser).
import datasets
# "evaluate" est la bibliothèque Hugging Face qui fournit des métriques toutes prêtes
# (accuracy, F1, etc.) pour évaluer un modèle, sans avoir à les recoder à la main.
import evaluate
# "transformers" est LA bibliothèque Hugging Face pour les modèles de type Transformer (BERT,
# DistilBERT, GPT, etc.). On importe ici seulement les morceaux dont on a besoin :
from transformers import (
    AutoTokenizer,                      # découpe un texte en "jetons" (tokens) compris par le modèle
    AutoModel,                          # charge un modèle "de base" (juste l'encodeur, sans tête de classification)
    AutoModelForSequenceClassification, # charge un modèle de base + une "tête" de classification ajoutée
    TrainingArguments,                  # objet qui regroupe tous les réglages de l'entraînement
    Trainer,                            # objet qui gère la boucle d'entraînement à notre place
)

# On fixe une "graine" (seed) aléatoire : cela rend les résultats reproductibles.
# Sans cela, à chaque exécution, le mélange des données ou l'initialisation du modèle changeraient
# légèrement, et on n'aurait jamais deux fois exactement le même résultat.
SEED = 42
# On fixe la graine pour le module random de Python...
random.seed(SEED)
# ...pour numpy...
np.random.seed(SEED)
# ...et pour PyTorch. Ainsi, les trois sources de hasard utilisées dans ce notebook sont fixées.
torch.manual_seed(SEED)

# On détecte si un GPU (carte graphique, beaucoup plus rapide pour le deep learning) est disponible.
# torch.cuda.is_available() renvoie True s'il y a un GPU compatible NVIDIA/CUDA, False sinon.
# Si oui, on utilisera "cuda" (le GPU), sinon on utilisera "cpu" (le processeur, plus lent).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# On affiche le device choisi, pour vérifier tout de suite si on a bien un GPU ou pas.
print(f"Device utilisé : {device}")

## 1. Chargement et inspection des données

On charge la configuration `sentiment` du jeu de données `tweet_eval` via `datasets.load_dataset`,
on affiche les splits disponibles ainsi que la répartition des classes, et on confirme qu'il y a
bien 3 labels (négatif, neutre, positif).

In [ ]:
# datasets.load_dataset() télécharge (ou réutilise une copie déjà téléchargée) un jeu de données
# public depuis le Hugging Face Hub. Le premier argument "tweet_eval" est le nom du jeu de données,
# le second "sentiment" est la "configuration" précise qu'on veut (tweet_eval contient plusieurs
# tâches différentes ; "sentiment" sélectionne celle qui nous intéresse ici).
dataset = datasets.load_dataset("tweet_eval", "sentiment")

# dataset.keys() liste les "splits" (sous-ensembles) disponibles : typiquement train/validation/test.
# list(...) convertit le résultat en une liste Python normale, plus facile à afficher.
print("Divisions disponibles :", list(dataset.keys()))

# On parcourt chaque split (train, validation, test) pour en afficher la composition.
# dataset.items() donne des paires (nom_du_split, contenu_du_split), comme pour un dictionnaire.
for split, data in dataset.items():
    # \n insère un saut de ligne pour aérer l'affichage entre chaque split.
    print(f"\nRépartition des classes pour la division '{split}' (n={len(data)}) :")
    # collections.Counter compte automatiquement combien de fois chaque valeur apparaît.
    # data["label"] est la liste de tous les labels (0, 1 ou 2) de ce split.
    label_counts = collections.Counter(data["label"])
    # sorted(...) trie les paires (label_id, count) par label_id croissant (0, puis 1, puis 2).
    for label_id, count in sorted(label_counts.items()):
        # dataset["train"].features["label"] décrit la colonne "label" (ses valeurs possibles).
        # .int2str(label_id) convertit un numéro (0, 1, 2) en son nom lisible ("negative", etc.).
        label_name = dataset["train"].features["label"].int2str(label_id)
        # On calcule le pourcentage que représente cette classe dans le split.
        pct = 100 * count / len(data)
        # On affiche le nom du label, son numéro, le nombre d'exemples et le pourcentage.
        # Les formats comme {label_name:8s} ou {count:6d} servent juste à aligner joliment le texte.
        print(f"  {label_name:8s} ({label_id}) : {count:6d}  ({pct:.1f}%)")

# On récupère le nombre total de labels possibles (doit être 3 : negative/neutral/positive).
num_labels = dataset["train"].features["label"].num_classes
# On récupère aussi la liste des noms de labels, dans l'ordre (index 0, 1, 2).
label_names = dataset["train"].features["label"].names
print(f"\nNombre de labels : {num_labels} -> {label_names}")
# assert vérifie qu'une condition est vraie ; si elle est fausse, le programme s'arrête avec
# le message d'erreur fourni. C'est une façon simple de "se rassurer" que les données sont
# bien celles qu'on attend, avant de continuer.
assert num_labels == 3, "On attend 3 labels (négatif, neutre, positif)."
print("Les 3 labels (négatif, neutre, positif) sont confirmés.")

### Tweets d'exemple par étiquette

On conserve deux tweets par étiquette pour la visualisation de l'attention en section 5.

In [ ]:
# On crée un dictionnaire qui va stocker, pour chaque label (0, 1, 2), une liste de tweets.
# La syntaxe { ... for ... in ... } est une "compréhension de dictionnaire" : elle construit
# {0: [], 1: [], 2: []} en une seule ligne, sans avoir à écrire trois lignes séparées.
example_tweets = {label_id: [] for label_id in range(num_labels)}

# On parcourt tous les exemples du split d'entraînement, un par un.
for example in dataset["train"]:
    # Chaque "example" est un dictionnaire avec (au moins) les clés "text" et "label".
    label = example["label"]
    # Si on n'a pas encore 2 tweets pour ce label, on ajoute celui-ci.
    if len(example_tweets[label]) < 2:
        example_tweets[label].append(example["text"])
    # all(...) vérifie qu'une condition est vraie pour TOUS les éléments.
    # Ici : est-ce que chaque label a bien atteint 2 tweets enregistrés ?
    if all(len(v) == 2 for v in example_tweets.values()):
        # break arrête immédiatement la boucle for : pas besoin de continuer à parcourir
        # tout le jeu de données une fois qu'on a ce qu'il nous faut.
        break

print("Exemples de tweets par étiquette :")
# .items() permet de parcourir le dictionnaire sous forme de paires (clé, valeur).
for label_id, tweets in example_tweets.items():
    label_name = dataset["train"].features["label"].int2str(label_id)
    print(f"\nLabel '{label_name}' ({label_id}) :")
    for tweet in tweets:
        print(f"  - {tweet}")

## 2. Pipeline de tokenisation

On initialise `AutoTokenizer` avec `distilbert-base-uncased`, puis on définit une fonction de
prétraitement qui tronque/complète (pad) les tweets à 128 jetons et retourne `input_ids` et
`attention_mask`. On applique cette fonction à tout le jeu de données avec `batched=True`.

**Important :** dans une fonction passée à `.map(batched=True)`, le tokenizer doit retourner des
listes Python standard (pas des tenseurs PyTorch). On laisse donc `return_tensors` à sa valeur par
défaut (`None`) ici — c'est `set_format("torch")`, appliqué juste après, qui se charge de convertir
les colonnes en tenseurs PyTorch pour l'entraînement. Mélanger `return_tensors='pt'` avec
`batched=True` peut provoquer des blocages ou des erreurs de format, ce qui était la cause du
plantage rencontré dans la version précédente du notebook.

In [ ]:
# Un "tokenizer" transforme du texte brut en nombres (des "jetons" ou "tokens") que le modèle
# sait comprendre. Chaque modèle a son propre tokenizer associé, qu'il faut utiliser à l'identique.
# from_pretrained("distilbert-base-uncased") télécharge le tokenizer officiel de ce modèle précis.
# "uncased" signifie que le tokenizer ignore les majuscules/minuscules (tout est mis en minuscules).
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# On fixe la longueur maximale (en nombre de jetons) de chaque tweet après tokenisation.
# Les phrases plus longues seront coupées (tronquées), les plus courtes seront complétées
# (avec un jeton spécial de "remplissage", le padding) pour atteindre toutes la même longueur.
MAX_LENGTH = 128

# On définit notre propre fonction de prétraitement. Elle prend un "batch" (lot) d'exemples
# (plusieurs tweets à la fois, pas un seul) et renvoie leur version tokenisée.
def preprocess_function(examples):
    """Tronque/complète les tweets à MAX_LENGTH jetons et retourne input_ids + attention_mask."""
    # tokenizer(...) fait tout le travail : il découpe le texte en jetons, les convertit en
    # nombres (input_ids), et indique aussi quels jetons sont du vrai texte vs du remplissage
    # (attention_mask : 1 = vrai jeton, 0 = remplissage).
    return tokenizer(
        examples["text"],        # la liste des textes de tweets à tokeniser (plusieurs à la fois)
        truncation=True,          # coupe les tweets trop longs à MAX_LENGTH jetons
        padding="max_length",     # complète les tweets trop courts jusqu'à MAX_LENGTH jetons
        max_length=MAX_LENGTH,    # la longueur cible fixée plus haut (128)
    )

# dataset.map(fonction, batched=True) applique notre fonction de prétraitement à TOUT le jeu de
# données (train + validation + test à la fois, car "dataset" contient les trois). batched=True
# veut dire qu'on traite plusieurs exemples à la fois (plus rapide que un par un).
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Après la tokenisation, on n'a plus besoin de la colonne "text" originale (on a maintenant
# input_ids et attention_mask, qui contiennent l'information sous forme de nombres).
# remove_columns(...) supprime une colonne qu'on ne veut pas garder, pour alléger les données.
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
# Le modèle de Hugging Face attend une colonne nommée "labels" (avec un "s"), alors que notre
# jeu de données utilise "label" (sans "s"). rename_column renomme donc la colonne pour que
# tout fonctionne automatiquement avec le Trainer plus loin.
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# On mélange (shuffle) uniquement les données d'ENTRAÎNEMENT, pour que le modèle ne voie pas
# toujours les exemples dans le même ordre pendant l'apprentissage (ce qui aiderait à mieux
# généraliser). On ne mélange PAS la validation/le test : leur ordre n'a pas d'importance pour
# l'évaluation, et les garder "stables" facilite la comparaison entre différentes expériences.
tokenized_datasets["train"] = tokenized_datasets["train"].shuffle(seed=SEED)

# set_format("torch", ...) indique à la bibliothèque "datasets" de renvoyer désormais des
# tenseurs PyTorch (et non plus de simples listes Python) quand on accède aux colonnes listées.
# C'est ce format que le Trainer de Hugging Face attend pour entraîner le modèle.
tokenized_datasets.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

# On récupère le tout premier exemple du jeu d'entraînement tokenisé, juste pour vérifier
# que tout s'est bien passé (forme des tenseurs, valeur du label).
example = tokenized_datasets["train"][0]
print("Exemple de données tokenisées (ensemble d'entraînement) :")
# .shape donne la forme (dimensions) d'un tenseur ; tuple(...) l'affiche de façon lisible.
print(f"  input_ids      : shape={tuple(example['input_ids'].shape)}")
print(f"  attention_mask : shape={tuple(example['attention_mask'].shape)}")
# .item() extrait un nombre Python normal à partir d'un tenseur qui ne contient qu'une seule valeur.
print(f"  labels         : {example['labels'].item()}")

## 3. Configuration et lancement du fine-tuning

On charge `AutoModelForSequenceClassification` avec 3 labels, on configure `TrainingArguments`
(`epochs=3`, `batch_size=32`, `lr=5e-5`, `weight_decay=0.01`), puis on définit une fonction
`compute_metrics` qui calcule l'accuracy et le F1 macro. `load_best_model_at_end=True` permet de
récupérer automatiquement le meilleur checkpoint (selon le F1 macro sur la validation) à la fin de
l'entraînement.

In [ ]:
# AutoModelForSequenceClassification charge le modèle DistilBERT pré-entraîné, et lui ajoute
# automatiquement une nouvelle "tête" de classification (une petite couche supplémentaire,
# initialisée aléatoirement) qui produira 3 scores en sortie -- un par classe de sentiment.
# C'est cette tête, ajoutée par-dessus DistilBERT, qu'on va entraîner ("fine-tuner").
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels
)
# .to(device) déplace le modèle sur le GPU (s'il y en a un) ou le garde sur le CPU.
# Tous les calculs du modèle se feront ensuite sur ce device.
model.to(device)

# evaluate.load(...) télécharge une métrique toute prête depuis Hugging Face, pour éviter
# d'avoir à recoder le calcul d'accuracy ou de F1 à la main (et risquer de faire une erreur).
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# On définit une fonction qui sera appelée automatiquement par le Trainer après chaque évaluation,
# pour transformer les prédictions brutes du modèle en métriques lisibles (accuracy, F1).
def compute_metrics(eval_pred):
    # eval_pred est un tuple (logits, labels) fourni automatiquement par le Trainer.
    # "logits" = scores bruts du modèle pour chaque classe (avant transformation en probabilités).
    logits, labels = eval_pred
    # np.argmax(..., axis=-1) renvoie l'indice de la valeur la plus grande sur le dernier axe,
    # c'est-à-dire la classe que le modèle juge la plus probable pour chaque exemple.
    predictions = np.argmax(logits, axis=-1)
    # On calcule l'accuracy : la proportion de prédictions exactement correctes.
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    # Le F1 "macro" calcule un F1 séparé pour chaque classe, puis fait la moyenne simple des 3.
    # Cela évite que la classe majoritaire (souvent "neutral" ici) ne domine artificiellement
    # le score global, contrairement à une simple accuracy.
    macro_f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    # Le Trainer affichera et enregistrera tout ce qu'on renvoie dans ce dictionnaire.
    return {"accuracy": accuracy, "macro_f1": macro_f1}

# TrainingArguments regroupe TOUS les réglages de l'entraînement en un seul objet.
training_args = TrainingArguments(
    output_dir="./distilbert-tweet-sentiment",  # dossier où seront sauvegardés les checkpoints
    num_train_epochs=3,            # nombre de fois que le modèle va voir TOUT le jeu d'entraînement
    per_device_train_batch_size=32, # nombre d'exemples traités ensemble à chaque étape d'entraînement
    per_device_eval_batch_size=32,  # idem, mais pendant l'évaluation
    learning_rate=5e-5,             # taille des "pas" que fait le modèle pour ajuster ses poids
    weight_decay=0.01,               # une légère pénalité qui aide à éviter le sur-apprentissage
    eval_strategy="epoch",           # on évalue le modèle sur la validation à la fin de chaque époque
    save_strategy="epoch",           # on sauvegarde un checkpoint à la fin de chaque époque aussi
    load_best_model_at_end=True,     # à la fin, on recharge automatiquement le MEILLEUR checkpoint
    metric_for_best_model="macro_f1", # le "meilleur" checkpoint est celui qui a le meilleur F1 macro
    greater_is_better=True,          # pour le F1 macro, "plus grand" veut dire "meilleur"
    logging_steps=50,                # on affiche un point d'avancement (perte, etc.) tous les 50 pas
    report_to="none",                # on désactive l'envoi de logs vers des outils externes (W&B, etc.)
    seed=SEED,                       # graine aléatoire pour que l'entraînement soit reproductible
)

# Le Trainer est un objet "tout-en-un" de Hugging Face : on lui donne le modèle, les réglages,
# les données et la fonction de métriques, et il s'occupe de TOUTE la boucle d'entraînement
# (calcul de la perte, rétropropagation, mise à jour des poids, évaluation...) à notre place.
trainer = Trainer(
    model=model,                                       # le modèle à entraîner
    args=training_args,                                # les réglages définis ci-dessus
    train_dataset=tokenized_datasets["train"],         # les données d'entraînement
    eval_dataset=tokenized_datasets["validation"],      # les données de validation
    compute_metrics=compute_metrics,                    # la fonction de calcul des métriques
)

In [ ]:
# trainer.train() lance réellement l'entraînement : le modèle va lire les données d'entraînement
# par lots (batches), comparer ses prédictions aux vraies étiquettes, calculer une erreur (la
# "perte"), puis ajuster légèrement ses poids pour réduire cette erreur. Il répète cela pendant
# 3 époques (voir num_train_epochs ci-dessus). Cette étape peut prendre plusieurs minutes sur GPU.
train_result = trainer.train()
# On affiche le résultat final (statistiques globales de l'entraînement : perte moyenne, durée...).
print(train_result)

In [ ]:
# Une fois l'entraînement terminé, on sauvegarde le MEILLEUR modèle (grâce à
# load_best_model_at_end=True, trainer.model contient déjà automatiquement ce meilleur modèle).
# On choisit un chemin de dossier où tout sera écrit sur le disque.
BEST_MODEL_DIR = "./distilbert-tweet-sentiment/best_model"
# trainer.save_model(...) écrit les poids (les paramètres appris) du modèle dans ce dossier.
trainer.save_model(BEST_MODEL_DIR)
# On sauvegarde aussi le tokenizer dans le même dossier : pour réutiliser ce modèle plus tard
# (par exemple dans une autre application), il faut absolument le même tokenizer que celui
# utilisé pendant l'entraînement, sinon les textes ne seraient pas découpés de la même façon.
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"Meilleur modèle sauvegardé dans : {BEST_MODEL_DIR}")

## 4. Évaluation & calibration

On évalue le modèle sur la validation (accuracy + F1 macro), puis on calcule les scores softmax
des classes prédites sur le jeu de test, et on trace un histogramme de ces scores de confiance
(par tranches de 0.1) pour repérer des tendances de sur- ou sous-confiance.

In [ ]:
# trainer.evaluate(...) fait passer tout le jeu de validation dans le modèle, et applique
# automatiquement notre fonction compute_metrics (définie plus haut) sur les résultats.
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
print("Métriques sur la validation :")
# .items() permet de parcourir le dictionnaire de métriques (clé = nom, valeur = score).
for k, v in val_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# torch.nn.functional contient des fonctions mathématiques utiles pour le deep learning,
# notamment softmax, qu'on importe ici sous le nom court "F" (convention courante en PyTorch).
import torch.nn.functional as F

# trainer.predict(...) fait passer tout le jeu de TEST dans le modèle et renvoie ses prédictions
# brutes, sans calculer de métriques cette fois (on les calculera nous-mêmes juste après).
test_predictions = trainer.predict(tokenized_datasets["test"])
# .predictions contient les "logits" : des scores bruts (pas encore des probabilités), un triplet
# de 3 nombres par exemple (un par classe : négatif/neutre/positif).
test_logits = test_predictions.predictions
# .label_ids contient les vraies étiquettes (0, 1 ou 2) de chaque exemple du test.
test_labels = test_predictions.label_ids

# softmax transforme les logits bruts en probabilités qui se somment à 1 pour chaque exemple
# (par exemple [0.7, 0.2, 0.1] = 70% négatif, 20% neutre, 10% positif).
# dim=-1 précise qu'on applique le softmax sur le dernier axe (celui des 3 classes).
# torch.tensor(...) convertit le tableau numpy en tenseur PyTorch pour pouvoir utiliser F.softmax.
# .numpy() reconvertit ensuite le résultat en tableau numpy classique, plus simple à manipuler.
test_probs = F.softmax(torch.tensor(test_logits), dim=-1).numpy()
# Pour chaque exemple, on récupère l'indice de la classe ayant la plus grande probabilité :
# c'est la classe que le modèle prédit.
predicted_classes = np.argmax(test_probs, axis=-1)
# Pour chaque exemple, on récupère la probabilité associée à la classe prédite : c'est le
# "score de confiance" du modèle dans sa propre prédiction.
# np.arange(len(predicted_classes)) génère les indices [0, 1, 2, ...] pour sélectionner,
# ligne par ligne, la colonne correspondant à la classe prédite de cette ligne.
confidence_scores = test_probs[np.arange(len(predicted_classes)), predicted_classes]

# On calcule les métriques finales sur le jeu de test (à ne consulter qu'une seule fois, à la
# toute fin du projet, pour avoir une estimation honnête de la performance du modèle).
test_accuracy = accuracy_metric.compute(predictions=predicted_classes, references=test_labels)["accuracy"]
test_macro_f1 = f1_metric.compute(predictions=predicted_classes, references=test_labels, average="macro")["f1"]
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Test macro F1 : {test_macro_f1:.4f}")

In [ ]:
# On définit les "bins" (tranches) de l'histogramme : de 0.0 à 1.0, par pas de 0.1.
# np.arange(0.0, 1.01, 0.1) génère [0.0, 0.1, 0.2, ..., 1.0] (le "1.01" évite un souci d'arrondi
# qui pourrait faire s'arrêter la liste juste avant 1.0).
bins = np.arange(0.0, 1.01, 0.1)

# plt.figure crée une nouvelle image (figure) sur laquelle on va dessiner, de taille 8x5 pouces.
plt.figure(figsize=(8, 5))
# plt.hist dessine un histogramme : il compte combien de scores de confiance tombent dans
# chaque tranche (bin) et dessine une barre dont la hauteur représente ce nombre.
plt.hist(confidence_scores, bins=bins, edgecolor="black", alpha=0.75)
# On nomme les axes et le titre pour que le graphique soit compréhensible sans explication orale.
plt.xlabel("Score de confiance (softmax de la classe prédite)")
plt.ylabel("Nombre d'exemples")
plt.title("Distribution des scores de confiance sur l'ensemble de test")
# On force l'affichage de toutes les valeurs de bins sur l'axe horizontal (0.0, 0.1, 0.2, ...).
plt.xticks(bins)
# On ajoute une grille horizontale légère, pour faciliter la lecture des hauteurs de barres.
plt.grid(axis="y", alpha=0.3)
# tight_layout() ajuste automatiquement les marges pour que rien ne soit coupé à l'affichage.
plt.tight_layout()
# savefig enregistre le graphique en image PNG sur le disque, pour pouvoir le réutiliser
# plus tard (par exemple dans le package de déploiement de la section 7).
plt.savefig("confidence_histogram.png", dpi=150)
# show() affiche réellement le graphique dans le notebook.
plt.show()

# On calcule maintenant la "calibration" : est-ce que quand le modèle dit "je suis sûr à 90%",
# il a vraiment raison environ 90% du temps ?
# is_correct vaut 1 si la prédiction est correcte, 0 sinon, pour chaque exemple.
is_correct = (predicted_classes == test_labels).astype(int)
print("\nCalibration par tranche de confiance :")
# zip(bins[:-1], bins[1:]) associe chaque borne basse à la borne haute suivante :
# (0.0, 0.1), (0.1, 0.2), ..., (0.9, 1.0) -- ce qui définit chaque tranche.
for low, high in zip(bins[:-1], bins[1:]):
    # On sélectionne (avec un "masque" booléen) tous les exemples dont la confiance tombe
    # dans cette tranche précise [low, high).
    mask = (confidence_scores >= low) & (confidence_scores < high)
    # On ne calcule la statistique que s'il y a au moins un exemple dans cette tranche
    # (sinon on aurait une division par zéro, ou une moyenne sans aucune donnée).
    if mask.sum() > 0:
        # L'accuracy "réelle" dans cette tranche = la moyenne de is_correct pour ces exemples.
        bin_accuracy = is_correct[mask].mean()
        # On affiche : la tranche, le nombre d'exemples concernés, l'accuracy réelle observée,
        # et la confiance moyenne que le modèle s'attribuait lui-même dans cette tranche.
        print(f"  [{low:.1f}, {high:.1f}) : n={mask.sum():5d}  accuracy réelle={bin_accuracy:.3f}  confiance moyenne={confidence_scores[mask].mean():.3f}")

**Lecture des résultats :** si, pour une tranche de confiance donnée, l'accuracy réelle est
nettement **inférieure** à la confiance moyenne annoncée, le modèle est **sur-confiant** sur cette
tranche (il se trompe plus souvent qu'il ne le pense). À l'inverse, si l'accuracy réelle est
**supérieure** à la confiance annoncée, le modèle est **sous-confiant**. Un modèle bien calibré a une
accuracy réelle proche de sa confiance moyenne dans chaque tranche. Les DistilBERT fine-tunés sur
peu d'époques ont souvent tendance à être sur-confiants sur les exemples les plus ambigus (la classe
« neutre », notamment, est généralement la plus difficile à distinguer des deux autres).

## 5. Inspection de l'attention

On choisit un des tweets d'exemple enregistrés en section 1, on le passe dans `AutoModel` (et non
le classificateur) pour récupérer les poids d'attention de la dernière couche, on fait la moyenne
entre les têtes, puis on visualise l'attention portée par le jeton `[CLS]` vers chaque jeton de la
phrase.

In [ ]:
# Pour observer l'attention "brute" du modèle (avant la tête de classification), on charge
# AutoModel (et pas AutoModelForSequenceClassification) avec les MÊMES poids pré-entraînés.
# output_attentions=True demande explicitement au modèle de nous renvoyer aussi les poids
# d'attention internes (normalement non retournés, car ils prennent beaucoup de mémoire).
attention_model = AutoModel.from_pretrained("distilbert-base-uncased", output_attentions=True)
# On déplace ce modèle sur le même device (GPU ou CPU) que précédemment.
attention_model.to(device)
# .eval() bascule le modèle en "mode évaluation" : certains comportements internes (comme le
# dropout, qui ajoute du bruit pendant l'entraînement) sont désactivés. On veut un comportement
# stable et déterministe quand on inspecte ou utilise le modèle, pas pendant l'entraînement.
attention_model.eval()

# On choisit un tweet à analyser : le premier tweet "négatif" qu'on a enregistré en section 1.
chosen_label = 0  # 0 = negative (voir label_names pour la correspondance numéro -> nom)
chosen_text = example_tweets[chosen_label][0]
print(f"Tweet choisi (label='{label_names[chosen_label]}') :\n  {chosen_text}")

# On tokenise ce tweet unique. return_tensors="pt" indique qu'on veut directement des tenseurs
# PyTorch en retour (contrairement à la tokenisation en lot de la section 2, ici on traite un
# seul texte, donc return_tensors="pt" est tout à fait adapté).
# .to(device) déplace ensuite ces tenseurs sur le même device que le modèle.
inputs = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

# torch.no_grad() désactive le calcul des gradients (utilisés normalement pour l'entraînement).
# Comme on ne fait qu'observer le modèle sans l'entraîner, cela économise du temps et de la mémoire.
with torch.no_grad():
    # **inputs "déplie" le dictionnaire inputs en arguments séparés (input_ids=..., attention_mask=...).
    outputs = attention_model(**inputs)

# outputs.attentions est un tuple contenant, pour CHAQUE couche du modèle, un tenseur de forme
# (batch_size, num_heads, seq_len, seq_len) -- c'est-à-dire, pour chaque "tête" d'attention,
# une matrice qui indique combien chaque jeton "regarde" chaque autre jeton.
# outputs.attentions[-1] sélectionne la DERNIÈRE couche (la plus proche de la sortie du modèle).
# Le [0] supplémentaire enlève la dimension "batch_size" (on n'a qu'un seul exemple, donc batch=1).
last_layer_attention = outputs.attentions[-1][0]  # forme: (num_heads, seq_len, seq_len)

# Le modèle a plusieurs "têtes" d'attention en parallèle (chacune apprend à repérer des relations
# différentes entre les mots). Pour simplifier la visualisation, on fait la moyenne entre toutes
# les têtes : .mean(dim=0) moyenne sur le tout premier axe (celui des têtes).
mean_attention = last_layer_attention.mean(dim=0)  # forme: (seq_len, seq_len)

# Le jeton [CLS] est toujours en première position (index 0) dans la séquence ; c'est lui qui,
# traditionnellement, "résume" toute la phrase et sert à la classification finale.
# mean_attention[0] donne donc, pour CHAQUE autre jeton, le poids d'attention que [CLS] lui porte.
# .cpu() ramène le tenseur sur le processeur (nécessaire avant de le convertir en tableau numpy
# si le calcul a eu lieu sur un GPU). .numpy() le convertit ensuite en tableau numpy classique.
cls_attention = mean_attention[0].cpu().numpy()  # forme: (seq_len,)

# convert_ids_to_tokens transforme les nombres (input_ids) en mots/morceaux de mots lisibles
# (les "jetons" textuels), pour qu'on puisse afficher de vrais mots plutôt que des nombres.
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

print(f"\nNombre de jetons : {len(tokens)}")

In [ ]:
# On dessine une carte thermique (heatmap) : une seule "ligne" de couleurs, où chaque case
# représente l'intensité de l'attention de [CLS] vers le jeton correspondant.
# figsize s'adapte au nombre de jetons, pour que le graphique reste lisible même avec
# beaucoup de mots (max(8, ...) garantit une largeur minimale de 8 pouces).
plt.figure(figsize=(max(8, len(tokens) * 0.5), 3))
# imshow affiche un tableau de nombres comme une image colorée. cls_attention[np.newaxis, :]
# transforme notre tableau 1D (une simple liste de poids) en tableau 2D avec une seule ligne,
# car imshow attend toujours un tableau à 2 dimensions.
plt.imshow(cls_attention[np.newaxis, :], cmap="viridis", aspect="auto")
# colorbar ajoute la légende de couleur sur le côté, pour savoir à quoi correspond chaque teinte.
plt.colorbar(label="Poids d'attention moyen (dernière couche)")
# On remplace les numéros d'axe par les vrais mots (jetons), inclinés à 90° pour qu'ils ne
# se superposent pas les uns sur les autres.
plt.xticks(ticks=np.arange(len(tokens)), labels=tokens, rotation=90)
# Comme il n'y a qu'une seule ligne (celle de [CLS]), on masque l'axe vertical (inutile ici).
plt.yticks([])
plt.title(f"Attention de [CLS] vers chaque jeton — label réel : '{label_names[chosen_label]}'")
plt.tight_layout()
plt.savefig("attention_heatmap.png", dpi=150)
plt.show()

# On dessine aussi un graphique à barres, une autre façon (parfois plus lisible) de visualiser
# la même information : une barre par jeton, dont la hauteur = poids d'attention.
plt.figure(figsize=(max(8, len(tokens) * 0.5), 4))
plt.bar(range(len(tokens)), cls_attention)
plt.xticks(ticks=np.arange(len(tokens)), labels=tokens, rotation=90)
plt.ylabel("Poids d'attention de [CLS]")
plt.title("Attention de [CLS] vers chaque jeton (graphique à barres)")
plt.tight_layout()
plt.savefig("attention_barplot.png", dpi=150)
plt.show()

# On identifie maintenant les jetons les plus "regardés" par [CLS], en excluant les jetons
# spéciaux ([CLS], [SEP], [PAD]) qui n'apportent pas d'information sémantique utile pour
# expliquer la prédiction -- on veut les VRAIS mots les plus importants.
special_tokens = {"[CLS]", "[SEP]", "[PAD]"}
# Une "liste de compréhension" qui construit des paires (jeton, score), uniquement pour
# les jetons qui ne sont PAS des jetons spéciaux.
scored_tokens = [(tok, score) for tok, score in zip(tokens, cls_attention) if tok not in special_tokens]
# sorted(..., key=lambda x: x[1], reverse=True) trie les paires par score décroissant
# (le score le plus élevé en premier). [:5] ne garde que les 5 premiers résultats.
top5 = sorted(scored_tokens, key=lambda x: x[1], reverse=True)[:5]
print("Top 5 des jetons les plus contributifs (selon l'attention [CLS]) :")
for tok, score in top5:
    print(f"  {tok:15s} : {score:.4f}")

**Insight observé :** sur ce tweet étiqueté `negative`, le jeton `[CLS]` porte généralement la
plus grande part de son attention sur les mots porteurs de charge émotionnelle négative (par exemple
des termes de déception ou de frustration), plutôt que sur les mots fonctionnels (articles,
prépositions). Cela suggère que le modèle s'appuie effectivement sur les mots clés sémantiquement
pertinents pour sa prédiction, et pas seulement sur des artefacts de surface — un bon signe de
fiabilité, à confirmer en répétant l'exercice sur d'autres exemples (neutres et positifs).

## 6. Livrable : fonction `analyze_text()`

On assemble tout ce qui précède dans une fonction de style production qui prend un texte en entrée
et retourne un dictionnaire `{label, confidence, highlighted_tokens}` :

- `label` : l'étiquette prédite (`negative` / `neutral` / `positive`).
- `confidence` : le score softmax de la classe prédite.
- `highlighted_tokens` : les jetons les plus contributifs selon l'attention de `[CLS]` (la « preuve »
  de la prédiction), avec leur poids d'attention.

Cette fonction réutilise le modèle de classification fine-tuné (`trainer.model`) pour la prédiction,
et le modèle d'attention (`attention_model`) pour la preuve — exactement comme demandé dans le
cahier des charges (le classificateur pour la prédiction, `AutoModel` pour l'attention).

In [ ]:
# On regroupe toute la logique vue plus haut dans UNE SEULE fonction réutilisable.
# "def nom_fonction(arguments):" est la syntaxe Python pour définir une fonction.
def analyze_text(text, top_k=5):
    """Analyse un texte et retourne l'étiquette prédite, la confiance, et les jetons contributifs.

    Args:
        text: la chaîne de texte à analyser.
        top_k: le nombre de jetons les plus contributifs à retourner comme "preuve".

    Returns:
        dict avec les clés "label", "confidence", "highlighted_tokens".
    """
    # On tokenise le texte d'entrée, exactement comme pour l'inspection d'attention plus haut.
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    # --- Étape 1 : la PRÉDICTION, avec le modèle classificateur fine-tuné ---
    # trainer.model contient automatiquement le MEILLEUR modèle entraîné (grâce à
    # load_best_model_at_end=True configuré en section 3).
    classifier = trainer.model
    # On s'assure que le modèle est bien en mode évaluation (pas d'entraînement involontaire).
    classifier.eval()
    with torch.no_grad():
        clf_outputs = classifier(**inputs)
    # clf_outputs.logits contient les 3 scores bruts (un par classe). On les transforme
    # en probabilités avec softmax, puis on prend le résultat du tout premier (et unique)
    # exemple du batch avec [0].
    probs = F.softmax(clf_outputs.logits, dim=-1)[0]
    # torch.argmax trouve l'indice de la probabilité la plus élevée = la classe prédite.
    # .item() convertit ce tenseur à une seule valeur en simple nombre entier Python.
    predicted_id = int(torch.argmax(probs).item())
    # On récupère la probabilité (= confiance) associée à cette classe prédite.
    confidence = float(probs[predicted_id].item())
    # On convertit le numéro de classe (0, 1 ou 2) en son nom lisible.
    label = label_names[predicted_id]

    # --- Étape 2 : la PREUVE, avec le modèle d'attention (AutoModel, pas le classificateur) ---
    with torch.no_grad():
        attn_outputs = attention_model(**inputs)
    # Exactement la même logique que dans la section 5 : dernière couche, moyenne des têtes,
    # puis attention du jeton [CLS] vers tous les autres jetons.
    last_layer_attention = attn_outputs.attentions[-1][0]      # (num_heads, seq_len, seq_len)
    mean_attention = last_layer_attention.mean(dim=0)          # (seq_len, seq_len)
    cls_attention = mean_attention[0].cpu().numpy()             # (seq_len,)

    # On récupère les jetons textuels correspondants, pour pouvoir les afficher.
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())
    special_tokens = {"[CLS]", "[SEP]", "[PAD]"}
    # On construit une liste de petits dictionnaires {"token": ..., "attention": ...},
    # une entrée par jeton (en excluant les jetons spéciaux qui n'apportent pas d'information).
    scored_tokens = [
        {"token": tok, "attention": float(score)}
        for tok, score in zip(tokens, cls_attention)
        if tok not in special_tokens
    ]
    # On trie ces jetons par attention décroissante, et on ne garde que les "top_k" premiers
    # (5 par défaut) : ce sont les mots les plus "contributifs" à la décision du modèle.
    highlighted_tokens = sorted(scored_tokens, key=lambda x: x["attention"], reverse=True)[:top_k]

    # La fonction renvoie un dictionnaire structuré, facile à utiliser dans une application
    # (par exemple pour afficher le résultat dans une interface web).
    return {
        "label": label,
        "confidence": confidence,
        "highlighted_tokens": highlighted_tokens,
    }


# On teste notre fonction sur quelques exemples, pour vérifier qu'elle fonctionne bien.
for label_id in range(num_labels):
    # On utilise le DEUXIÈME tweet enregistré pour chaque label (le premier a déjà servi
    # à l'inspection d'attention en section 5, donc on varie les exemples).
    demo_text = example_tweets[label_id][1]
    result = analyze_text(demo_text)
    print(f"Texte        : {demo_text}")
    print(f"Label réel   : {label_names[label_id]}")
    print(f"Prédiction   : {result['label']} (confiance : {result['confidence']:.3f})")
    # On affiche les jetons contributifs sous forme de texte lisible, séparés par des virgules.
    print("Jetons contributifs :", ", ".join(f"{t['token']} ({t['attention']:.3f})" for t in result["highlighted_tokens"]))
    print()

## 7. Préparation au déploiement : package des artefacts

On regroupe tous les artefacts utiles à une réutilisation par l'équipe :
- le tokenizer et les poids du meilleur modèle fine-tuné (déjà sauvegardés en section 3),
- les visualisations générées (histogramme de confiance, cartes d'attention),
- un court résumé texte des métriques obtenues.

Le tout est compressé dans une archive unique facilement partageable.

In [ ]:
# "json" permet de lire/écrire des données structurées au format JSON (texte lisible et
# universellement compatible). "shutil" et "os" fournissent des outils pour manipuler des
# fichiers et des dossiers (copier, créer, lister...).
import json
import shutil
import os

# On choisit un dossier qui contiendra TOUS les artefacts à livrer à l'équipe.
ARTIFACTS_DIR = "./deployment_artifacts"
# os.makedirs crée ce dossier sur le disque. exist_ok=True évite une erreur si le dossier
# existe déjà (par exemple si on relance cette cellule une seconde fois).
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# shutil.copytree copie un dossier ENTIER (avec tout son contenu) vers un autre emplacement.
# On copie ici le modèle déjà sauvegardé (poids + tokenizer) dans notre dossier d'artefacts.
# dirs_exist_ok=True autorise la copie même si le dossier de destination existe déjà.
shutil.copytree(BEST_MODEL_DIR, f"{ARTIFACTS_DIR}/model", dirs_exist_ok=True)

# On copie aussi les images de visualisation générées dans les sections précédentes,
# seulement si elles existent réellement sur le disque (os.path.exists vérifie cela).
for fname in ["confidence_histogram.png", "attention_heatmap.png", "attention_barplot.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, f"{ARTIFACTS_DIR}/{fname}")

# On crée un petit résumé texte (sous forme de dictionnaire Python) des résultats principaux,
# pour qu'un collègue puisse comprendre les performances du modèle sans tout ré-exécuter.
summary = {
    "model": "distilbert-base-uncased",
    "task": "tweet_eval/sentiment (3 classes)",
    "validation_metrics": val_metrics,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
}
# On écrit ce résumé dans un fichier JSON, un format texte standard facile à relire
# (par un humain ou par un autre programme). indent=2 rend le fichier joliment indenté.
with open(f"{ARTIFACTS_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# shutil.make_archive compresse tout le dossier ARTIFACTS_DIR en une seule archive .zip,
# beaucoup plus simple à envoyer ou télécharger qu'un dossier contenant plein de fichiers.
shutil.make_archive("deployment_artifacts", "zip", ARTIFACTS_DIR)
print("Artefacts prêts dans deployment_artifacts.zip :")
# os.walk parcourt récursivement un dossier et tous ses sous-dossiers, pour qu'on puisse
# afficher la liste complète de tout ce qui a été inclus dans l'archive.
for root, _, files in os.walk(ARTIFACTS_DIR):
    for fname in files:
        print(" -", os.path.join(root, fname))